Lab 5: Linear Regression Neuron: Learning by Gradient
Descent (No ML Libraries)

In [7]:
import pandas as pd
import numpy as np

In [8]:
#PART A: DATA SETUP

# A1. Load dataset
# The abalone dataset from UCI doesn't have a header, so we add it manually.
columns = ["Sex", "Length", "Diameter", "Height", "Whole_weight",
           "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]
df = pd.read_csv("abalone.data", names=columns)

print(f"Dataset loaded with {df.shape[0]} rows.")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

# Checkpoint:
# what is input: Physical measurements of the abalone (Length, Diameter, Height).
# what is output: The predicted age of the abalone.
# why output is numeric: Because age is a continuous value on a ratio scale, not a category.

# A2. Convert target
df['y'] = df['Rings'] + 1.5

# A3. Choose features
# Justification: I am using Length, Diameter, and Height.
# These are basic physical dimensions that represent the size and volume of the abalone,
# which typically increase as the organism gets older.
X = df[["Length", "Diameter", "Height"]].values
y = df['y'].values.reshape(-1, 1)

# A4. Train-test split (Manual 80/20)
# Shuffling indices to ensure a random split.
indices = np.random.permutation(len(X))
split_idx = int(0.8 * len(X))

train_idx, test_idx = indices[:split_idx], indices[split_idx:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# A5. Normalize inputs
# We scale features so that no single measurement dominates the learning process.
train_mean = np.mean(X_train, axis=0)
train_std = np.std(X_train, axis=0)

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# Checkpoint:
# why normalization is needed: Features have different ranges. Normalization ensures
# stable gradients and prevents one feature from overpowering others.

Dataset loaded with 4177 rows.
Columns: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


In [9]:
#PART B: THE MODEL

# forward pass: y_hat = Xw + b
def forward(X, w, b):
    return np.dot(X, w) + b

# Initialize parameters: 3 weights for 3 features, 1 bias.
w = np.random.randn(3, 1) * 0.01
b = 0.0

# Checkpoint:
# parameters are: The weight coefficients (w1, w2, w3) and the bias (b).
# number of parameters: 4

In [10]:
# PART C: DEFINE LOSS (MSE)

def mse(y, y_hat):
    return np.mean((y - y_hat)**2)

# Checkpoint:
# why square: It removes the sign of the error and penalizes large mistakes more heavily.
# what mistakes are expensive: Outliers or large differences between true and predicted values.

In [11]:
# PART D: LEARNING RULE
# Checkpoint:
# what gradient means: The direction of steepest increase of the loss function.
# why subtracting: To move in the opposite direction (downhill) to minimize the error.

def grad_w(X, y, y_hat):
    #returns gradient with respect to w shape must match w: (d, 1)
    n = len(y)
    # Formula: (2/n) * X_transpose * (y_hat - y)
    dW = (2/n) * np.dot(X.T, (y_hat - y))

    # Constraint check: Shape must be (3, 1) if using 3 features
    return dW

def grad_b(y, y_hat):
    # returns gradient with respect to b
    # Formula: (2/n) * sum(y_hat - y)
    db = (2 / len(y)) * np.sum(y_hat - y)
    return db

# Checkpoint:
# meaning of large gradient: It means the model's current parameters are very far
# from the optimal values; the "slope" of the error surface is steep,
# indicating a need for a significant update.

# effect of too-large learning rate: The model might "overshoot" the minimum point
# of the loss function. Instead of converging, the loss might fluctuate
# or even increase (diverge) because the steps are too big.

In [12]:
# PART E: TRAINING LOOP

# Initialize w (small random values) and b (zero)
# d is the number of features (3)
w = np.random.randn(3, 1) * 0.01
b = 0.0

# Choose learning rate and epochs
lr = 0.01
epochs = 200

# Checkpoint:
# Initial expectation: I expect the loss to go down fast at the beginning
# because the initial random weights will have a high error, and the large
# gradients will cause significant corrections toward the correct direction.

for epoch in range(epochs):
    # 1) forward pass
    y_hat_train = forward(X_train, w, b)

    # 2) compute loss
    loss = mse(y_train, y_hat_train)

    # 3) compute gradients
    # Note: Using the functions defined in Part D
    dw = grad_w(X_train, y_train, y_hat_train)
    db = grad_b(y_train, y_hat_train)

    # 4) update w and b
    w = w - lr * dw
    b = b - lr * db

    # print progress every 20 epochs
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Checkpoint:
# Revised expectation after training: The loss decreased steadily and then
# started to level off (plateau). This confirms that the gradient descent
# correctly tuned the numbers to find the best possible fit for a linear equation.

Epoch 0, Loss: 141.0859
Epoch 20, Loss: 65.4621
Epoch 40, Loss: 32.8028
Epoch 60, Loss: 18.3545
Epoch 80, Loss: 11.9254
Epoch 100, Loss: 9.0603
Epoch 120, Loss: 7.7827
Epoch 140, Loss: 7.2126
Epoch 160, Loss: 6.9579
Epoch 180, Loss: 6.8437


In [13]:
# --- PART F: EVALUATION ---

y_pred = forward(X_test, w, b)
final_mse = mse(y_test, y_pred)
final_mae = np.mean(np.abs(y_test - y_pred))

print(f"\nTest MSE: {final_mse:.4f}")
print(f"Test MAE: {final_mae:.4f}")

print("\nSample Predictions:")
for i in range(5):
    true_val = float(y_test[i])
    pred_val = float(y_pred[i])
    print(f"True: {true_val:.1f}, Pred: {pred_val:.1f}, Abs Error: {abs(true_val - pred_val):.4f}")

# Checkpoint:
# systematic errors: The model struggles with older abalones where growth is non-linear.
# observed bias: As predicted in DLQ4, a single linear equation cannot bend to fit curved data.


Test MSE: 7.0379
Test MAE: 1.8339

Sample Predictions:
True: 8.5, Pred: 10.0, Abs Error: 1.4753
True: 12.5, Pred: 10.5, Abs Error: 2.0106
True: 8.5, Pred: 11.1, Abs Error: 2.5705
True: 9.5, Pred: 9.5, Abs Error: 0.0067
True: 11.5, Pred: 12.5, Abs Error: 0.9820


/tmp/ipykernel_255/3631408139.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  true_val = float(y_test[i])
/tmp/ipykernel_255/3631408139.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred_val = float(y_pred[i])


In [ ]:
#REFLECTIVE COMMENTS
# Difference from Perceptron:
# The perceptron used a hard step function to give a YES/NO label. This model
# uses a linear output to estimate a continuous numeric age.

# Why Sigmoid matters:
# In Lab 3, Sigmoid was essential to turn scores into probabilities (0 to 1).
# Here, we removed it because we need the raw numeric output for regression.

# The Unsolved Problem:
# Learning only "tuned the numbers" (w and b) within a fixed linear structure.
# Even with perfect gradient descent, the model cannot learn a curved
# relationship. To fix this, we need "intermediate representations" (multiple layers)
# so the model can change its shape, not just its position.